# Get LinkedIn Data via API

## Imports

In [ ]:
import requests
import json
import datetime as dt

## Parameters

In [ ]:
dbutils.widgets.combobox("run_type", "initial", ["initial", "delta"])
dbutils.widgets.text("catalog", "<your-dev-catalog>")
dbutils.widgets.text("schema", "bronze_linkedin")
dbutils.widgets.text("volume", "landing")

run_type = dbutils.widgets.get("run_type")
catalog  = dbutils.widgets.get("catalog")
schema   = dbutils.widgets.get("schema")
volume   = dbutils.widgets.get("volume")

volume_base = f"/Volumes/{catalog}/{schema}/{volume}"

In [ ]:
end_dt = dt.datetime.now() - dt.timedelta(days=1)
end = f"year:{end_dt.year},month:{end_dt.month},day:{end_dt.day}"

if run_type == "delta":
    start_dt = end_dt - dt.timedelta(days=60)
    start = f"year:{start_dt.year},month:{start_dt.month},day:{start_dt.day}"
else:
    start = f"year:2020,month:1,day:1"

In [ ]:
# Replace <your-secret-scope> with the name of your Databricks secret scope
# The secret key must hold a valid LinkedIn access token
access_token = dbutils.secrets.get("<your-secret-scope>", "linkedin-token")

## Functions

In [ ]:
def get_linkedin_data(url, access_token):
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json',
        'X-Restli-Protocol-Version': '2.0.0',
        'Linkedin-Version': '202506'
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()

    # LinkedIn can return error details with HTTP 200
    if "status" in data and data["status"] >= 400:
        raise RuntimeError(f"LinkedIn API error {data['status']}: {data.get('message', data)}")

    return data

In [ ]:
def write_data_to_json_file(data, path, file_prefix):
    timestamp = dt.datetime.now().strftime("%Y%m%d%H%M%S")
    file_path = f"{path}{file_prefix}{timestamp}.json"
    dbutils.fs.put(file_path, json.dumps(data), overwrite=True)

## Pull Data from LinkedIn

### Follower Deltas by Date

In [ ]:
url = f"https://api.linkedin.com/rest/memberFollowersCount?q=dateRange&dateRange=(start:({start}),end:({end}))"
data = get_linkedin_data(url, access_token)
write_data_to_json_file(data, f"{volume_base}/followers/", "followers_")

### Total Followers at the Moment

In [ ]:
url = f"https://api.linkedin.com/rest/memberFollowersCount?q=me"
data = get_linkedin_data(url, access_token)
write_data_to_json_file(data, f"{volume_base}/followers_agg/", "followers_")

### Post Impressions as Daily Aggregation

In [ ]:
url = f"https://api.linkedin.com/rest/memberCreatorPostAnalytics?q=me&queryType=IMPRESSION&aggregation=DAILY&dateRange=(start:({start}),end:({end}))"
data = get_linkedin_data(url, access_token)
write_data_to_json_file(data, f"{volume_base}/impressions/", "impressions_")

### Post Reactions as Daily Aggregation

In [ ]:
url = f"https://api.linkedin.com/rest/memberCreatorPostAnalytics?q=me&queryType=REACTION&aggregation=DAILY&dateRange=(start:({start}),end:({end}))"
data = get_linkedin_data(url, access_token)
write_data_to_json_file(data, f"{volume_base}/reactions/", "reactions_")

### Post Comments as Daily Aggregation

In [ ]:
url = f"https://api.linkedin.com/rest/memberCreatorPostAnalytics?q=me&queryType=COMMENT&aggregation=DAILY&dateRange=(start:({start}),end:({end}))"
data = get_linkedin_data(url, access_token)
write_data_to_json_file(data, f"{volume_base}/comments/", "comments_")